# Cropy 작업자 분리 시험셋: 병반 박스 crop 실험

448px 본 모델의 무검은무늬병 오류는 11월 중순 이후 촬영분과 피해가 심한 사진에 몰려 있습니다. 모델이 병반이 아니라 **배경과 잎 전체 상태(계절·촬영자 습관)** 를 단서로 쓰고 있다는 가설을 검증합니다.

AI Hub 라벨 JSON에는 질병 사진마다 병반 영역 박스(`annotations.points`, 원본 해상도 기준)가 있습니다. 이 박스를 512px 묶음 좌표로 축소해 **질병 사진은 박스 주변만 잘라 낸** 데이터셋을 만들고, 같은 작업자 분할·같은 설정(ResNet50, 448px, 8 epoch)으로 학습합니다. 정상·기타 사진은 박스가 없어 그대로 둡니다.

비교하는 네 조합:

| 학습 | 평가 | 의미 |
|---|---|---|
| 전체 사진 (`hires448_v1`, 기존) | 전체 사진 | 현재 성능 |
| 전체 사진 (기존) | 박스 crop | 배경을 치우면 현재 모델이 나아지는가 |
| 박스 crop (이번 학습) | 박스 crop | 병반만 보고 배우면 얼마나 가르는가 → 유도촬영(근접 재촬영) 흐름의 상한 |
| 박스 crop (이번 학습) | 전체 사진 | 첫 사진(박스 없음)에도 쓸 수 있는가 |

- 결과: `MyDrive/cropy_testset/boxcrop_v1/` (`best_model.pth`, `test_report.json`, `box_stats.json`, `predictions_*.csv`)
- A100이면 학습 30분 안팎, T4면 3시간입니다. 데이터 준비(1번)는 이전 실험과 같아 같은 세션이면 바로 넘어갑니다.

**GPU 런타임** 에서 위에서부터 차례로 실행하세요.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, re, shutil, subprocess, sys, zipfile
import numpy as np, torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
MY = Path('/content/drive/MyDrive')
TS = MY / 'cropy_testset'
RUN = TS / 'boxcrop_v1'
WORK = Path('/content/cropy_testset')
DATA = WORK / 'dataset'
SPLIT = WORK / 'split'
EPOCHS, BATCH_SIZE, LR, IMG_SIZE = 8, 16, 1e-4, 448
EXPECTED_BASE_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_CODE_SHA256 = 'efe7034c6acd0544416e0cdb94c8fe748081921d73cf2fd1513fbc9b55e292a4'
SOURCES = {'base': MY / 'cropy_full_512/cropy_full_512.zip', 'code': MY / 'cropy_other10/training_code_other10.zip',
           'aihub': MY / 'cropy_other10/aihub_other_v1.zip', 'coco': MY / 'cropy_other10/coco_other_v1.zip',
           'server': MY / 'cropy_other10/cropy_other10_handoff_20261027/full512_resnet50_other10_v1.pth',
           'workers': TS / 'workers.json', 'trainer': TS / 'train_full_colab_convnext.py',
           'baseline': TS / 'baseline_v1/best_model.pth', 'main448': TS / 'hires448_v1/best_model.pth'}
for path in SOURCES.values():
    assert path.is_file(), f'Drive에 파일이 없습니다: {path}'
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)

## 1. 데이터 준비와 작업자 단위 재분할
Drive에서 로컬로 복사해 풀고, 모든 사진을 작업자 배정표대로 `split/{train,val,test}/<클래스>/`에 옮깁니다. AI Hub 9클래스 사진 수가 설계와 정확히 같은지 확인합니다.

In [ ]:
EXPECT = {"train": {"0": 589, "1": 528, "2": 267, "3": 124, "4": 574, "5": 310, "6": 5932, "7": 6726, "8": 5724}, "val": {"0": 134, "1": 166, "2": 59, "3": 49, "4": 107, "5": 58, "6": 1440, "7": 2023, "8": 1688}, "test": {"0": 361, "1": 330, "2": 212, "3": 78, "4": 230, "5": 125, "6": 2313, "7": 3349, "8": 3087}}

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack(src, dest, expected=None):
    local = WORK / src.name
    if not local.exists():
        shutil.copyfile(src, local)
    if expected:
        assert digest(local) == expected, f'해시 불일치: {src.name}'
    with zipfile.ZipFile(local) as z:
        for item in z.infolist():
            assert (dest / item.filename).resolve().is_relative_to(dest.resolve()), '잘못된 압축 경로'
        z.extractall(dest)

unpack(SOURCES['code'], WORK / 'code', EXPECTED_CODE_SHA256)
ASSIGN = json.loads(SOURCES['workers'].read_text(encoding='utf-8'))['assign']
GROUP = re.compile(r'(?:^|_)([0-9A-Za-z]+)_(20\d{6})(?=_|\.|$)')
def split_of(path):
    m = GROUP.search(path.name)
    if m:
        return ASSIGN[m.group(1)] if m.group(1) in ASSIGN else 'train'
    if path.parent.name.startswith('9_'):  # COCO 조각: 내용 해시로 70/10/20
        b = int(digest(path)[:8], 16) % 10
        return 'train' if b < 7 else 'val' if b == 7 else 'test'
    return 'train'

if not (WORK / 'split_done').exists():
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.rmtree(SPLIT, ignore_errors=True)
    unpack(SOURCES['base'], WORK, EXPECTED_BASE_SHA256)
    (DATA / 'manifest.csv').rename(DATA / 'manifest_9class.csv')
    unpack(SOURCES['aihub'], WORK)
    unpack(SOURCES['coco'], WORK)
    rows = []
    for src_split in ('train', 'val'):
        for path in sorted((DATA / 'images' / src_split).glob('*/*')):
            s = split_of(path)
            dest = SPLIT / s / path.parent.name / path.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), dest)
            m = GROUP.search(path.name)
            rows.append([s, path.parent.name, path.name, m.group(1) if m else '', src_split])
    with (RUN / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['split', 'class', 'file', 'worker', 'official_split'])
        w.writerows(rows)
    (WORK / 'split_done').touch()

counts = {s: {c.name: len(list(c.iterdir())) for c in sorted((SPLIT / s).iterdir())} for s in ('train', 'val', 'test')}
for s, cs in counts.items():
    print(s, sum(cs.values()), cs)
for s, cs in EXPECT.items():
    for c, n in cs.items():
        got = counts[s][[k for k in counts[s] if k.startswith(c + '_')][0]]
        assert got == n, f'{s}/{c}: {got} != 설계 {n}'
print('AI Hub 9클래스 사진 수가 설계와 일치합니다.')
fingerprint = ''.join(f'{name}: {digest(WORK / SOURCES[name].name)}\n' for name in ('base', 'aihub', 'coco'))
fingerprint += f"workers: {digest(SOURCES['workers'])}\n"
(SPLIT.parent / 'manifest.csv').write_text(fingerprint, encoding='utf-8')  # 학습 스크립트는 data-dir 상위의 manifest.csv를 읽는다
from torchvision.datasets import ImageFolder
train_ds, val_ds = ImageFolder(SPLIT / 'train'), ImageFolder(SPLIT / 'val')
assert train_ds.classes == val_ds.classes and len(train_ds.classes) == 10, train_ds.classes
EXPECT_IMAGES = (len(train_ds), len(val_ds))
(RUN / 'split_counts.json').write_text(json.dumps(counts, ensure_ascii=False, indent=1), encoding='utf-8')
print('학습/검증:', EXPECT_IMAGES)

## 2. 병반 박스 crop 데이터셋 만들기
라벨 JSON의 박스를 512px 이미지 좌표로 축소하고, 각 변 15%씩 여유를 두어 잘라 냅니다. 박스가 여러 개면 합집합을 씁니다. 질병 사진인데 박스가 없거나 라벨이 없으면 원본을 그대로 둡니다.

In [ ]:
from PIL import Image
CROP = WORK / 'crop'
MARGIN = 0.15
label_index = {p.name[:-5]: p for p in (DATA / 'labels').rglob('*.json')}
print('라벨 JSON', len(label_index))

def box_of(name):
    j = label_index.get(name)
    if j is None:
        return None, 'no_label'
    d = json.loads(j.read_text(encoding='utf-8'))
    pts = (d.get('annotations') or {}).get('points') or []
    pts = [p for p in pts if all(k in p for k in ('xtl', 'ytl', 'xbr', 'ybr'))]
    if not pts:
        return None, 'no_points'
    W, H = d['description'].get('width'), d['description'].get('height')
    if not W or not H:  # 일부 라벨은 원본 크기가 0으로 기록돼 있어 박스를 축소할 수 없다
        return None, 'no_dims'
    x0, y0 = min(p['xtl'] for p in pts), min(p['ytl'] for p in pts)
    x1, y1 = max(p['xbr'] for p in pts), max(p['ybr'] for p in pts)
    return (x0 / W, y0 / H, x1 / W, y1 / H), 'ok'

stats = {'ok': 0, 'no_label': 0, 'no_points': 0, 'no_dims': 0, 'degenerate': 0, 'area_ratio': []}
if not (CROP / 'done').exists():
    shutil.rmtree(CROP, ignore_errors=True)
    for split in ('train', 'val', 'test'):
        for cls in sorted((SPLIT / split).iterdir()):
            out = CROP / 'images' / split / cls.name
            out.mkdir(parents=True, exist_ok=True)
            disease = cls.name[0] in '012345'
            for src in sorted(cls.iterdir()):
                dst = out / src.name
                rel, why = box_of(src.name) if disease else (None, 'normal')
                if rel is None:
                    if why in stats: stats[why] += 1
                    os.symlink(src, dst) if hasattr(os, 'symlink') else shutil.copyfile(src, dst)
                    continue
                with Image.open(src) as im:
                    im = im.convert('RGB')
                    w, h = im.size
                    bw, bh = (rel[2] - rel[0]) * w, (rel[3] - rel[1]) * h
                    if bw < 16 or bh < 16:
                        stats['degenerate'] += 1
                        im.save(dst, quality=95)
                        continue
                    l = max(0, int((rel[0] * w) - MARGIN * bw)); t = max(0, int((rel[1] * h) - MARGIN * bh))
                    r = min(w, int((rel[2] * w) + MARGIN * bw)); b = min(h, int((rel[3] * h) + MARGIN * bh))
                    stats['ok'] += 1
                    stats['area_ratio'].append(round((r - l) * (b - t) / (w * h), 4))
                    im.crop((l, t, r, b)).save(dst, quality=95)
    ar = np.array(stats['area_ratio'])
    stats['area_ratio'] = dict(n=int(len(ar)), p10=float(np.percentile(ar, 10)), p50=float(np.percentile(ar, 50)), p90=float(np.percentile(ar, 90)))
    (RUN / 'box_stats.json').write_text(json.dumps(stats, ensure_ascii=False, indent=1), encoding='utf-8')
    shutil.copyfile(SPLIT.parent / 'manifest.csv', CROP / 'manifest.csv')
    (CROP / 'done').touch()
print(json.loads((RUN / 'box_stats.json').read_text(encoding='utf-8')))
from torchvision.datasets import ImageFolder
crop_train, crop_val = ImageFolder(CROP / 'images/train'), ImageFolder(CROP / 'images/val')
assert (len(crop_train), len(crop_val)) == EXPECT_IMAGES and len(crop_train.classes) == 10
print('crop 데이터셋 학습/검증:', EXPECT_IMAGES)

## 3. 박스 crop으로 학습 (ImageNet에서 시작)
전체 사진 모델(`hires448_v1`)과 같은 설정이며 입력 폴더만 crop 데이터셋입니다. 메모리 부족이 나면 1번 셀의 BATCH_SIZE를 8로 줄이세요.

In [ ]:
import os
shutil.copyfile(SOURCES['trainer'], WORK / 'code/train_full_colab.py')
command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
    '--data-dir', str(CROP / 'images'), '--output-dir', str(RUN / 'main'),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR),
    '--num-workers', '4', '--img-size', str(IMG_SIZE), '--arch', 'resnet50', '--blur-aug',
    '--expect-images', str(EXPECT_IMAGES[0]), str(EXPECT_IMAGES[1])]
if (RUN / 'main/last_checkpoint.pth').exists():
    command.append('--resume')
print('이어 학습:', '--resume' in command)
subprocess.run(command, check=True)
shutil.copyfile(RUN / 'main/best_model.pth', RUN / 'best_model.pth')

## 4. 네 조합 평가
두 모델을 시험 작업자 사진의 전체 판과 crop 판에 각각 돌립니다. 10클래스 지표와 무 질병의 작업자·피해 정도·촬영일별 정답 수를 저장합니다.

In [ ]:
from torchvision import transforms

def load(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    sys.path.insert(0, str(WORK / 'code'))
    from train_full_colab import build_model
    model = build_model(ckpt.get('arch', 'resnet50'), ckpt['num_classes'], pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    return model.cuda().eval(), ckpt

def predict(model, size, files):
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    out = []
    for i in range(0, len(files), 64):
        batch = files[i:i + 64]
        with torch.no_grad():
            out.append(model(torch.stack([tf(Image.open(p).convert('RGB')) for p in batch]).cuda()).softmax(1).cpu().numpy())
    return np.concatenate(out)

def tok(p, k):
    t = p.name.split('_')
    return {'worker': t[8], 'date': t[9], 'risk': t[7]}[k] if len(t) > 9 else ''

def metrics(y, pred, ids, files):
    cm = np.zeros((len(ids), len(ids)), dtype=np.int64)
    for a_, b_ in zip(y, pred):
        cm[ids.index(a_), ids.index(b_)] += 1
    recall = cm.diagonal() / np.maximum(cm.sum(1), 1)
    precision = cm.diagonal() / np.maximum(cm.sum(0), 1)
    f1 = 2 * recall * precision / np.maximum(recall + precision, 1e-9)
    br = {}
    for key in ('worker', 'risk', 'date'):
        d = {}
        for p, yy, pp in zip(files, y, pred):
            if yy in (2, 3):
                k = f'{yy}/{tok(p, key)}'
                d.setdefault(k, [0, 0]); d[k][0] += int(pp == yy); d[k][1] += 1
        br[key] = {k: f'{v[0]}/{v[1]}' for k, v in sorted(d.items())}
    rad = np.isin(y, [2, 3])
    return dict(n=int(len(y)), accuracy=round(float((pred == y).mean()), 4), macro_f1=round(float(f1.mean()), 4),
                recall={str(c): round(float(r), 4) for c, r in zip(ids, recall)}, confusion=cm.tolist(),
                radish_disease_accuracy=round(float((pred[rad] == y[rad]).mean()), 4), radish_by=br)

views = {'full': sorted((SPLIT / 'test').glob('*/*')), 'crop': sorted((CROP / 'images/test').glob('*/*'))}
assert [p.name for p in views['full']] == [p.name for p in views['crop']]
y = np.array([int(p.parent.name.split('_')[0]) for p in views['full']])
report = {}
for mname, mpath in (('train_full', SOURCES['main448']), ('train_crop', RUN / 'best_model.pth')):
    model, ckpt = load(mpath)
    ids = [int(c.split('_', 1)[0]) for c in ckpt['classes']]
    for vname, files in views.items():
        probs = predict(model, ckpt.get('img_size', 256), files)
        pred = np.array(ids)[probs.argmax(1)]
        key = f'{mname}__eval_{vname}'
        report[key] = metrics(y, pred, ids, files)
        report[key]['epoch'] = int(ckpt['epoch'])
        with (RUN / f'predictions_{key}.csv').open('w', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            w.writerow(['file', 'label', 'worker'] + [f'p{i}' for i in ids])
            for p, pr in zip(files, probs):
                w.writerow([p.name, int(p.parent.name.split('_')[0]), tok(p, 'worker')] + [round(float(v), 5) for v in pr])
        r = report[key]
        print(f"{key:28s} 정확도 {r['accuracy']:.2%} F1 {r['macro_f1']:.3f} 검은무늬 {r['recall']['2']:.1%} 노균 {r['recall']['3']:.1%} 고추탄저 {r['recall']['0']:.1%} 배추검은썩음 {r['recall']['4']:.1%}")
    del model
(RUN / 'test_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
print('저장:', RUN / 'test_report.json')